# CampusVoice-RT: 평가셋 생성 노트북 (Colab)

로컬 디스크가 부족할 때 Colab에서 평가셋을 처음부터 다시 만든다. 모든 선택이 시드와 id 목록으로 고정되어 있어 결과 파일은 동일하다.

1. 원본 음성: Zeroth test 60 MB + FLEURS ko_kr test 215 MB (id 목록 `data/eval/clean_subset_ids_*.txt` 재사용)
2. 잡음: FSD50K 미러에서 선택 클립만 (수백 MB)
3. SNR 4조건 합성 → `data/eval/audio/`, `data/eval/manifest.jsonl`
4. Hugging Face Datasets 업로드 (토큰 필요)

In [ ]:
!git clone https://github.com/ekgP525/asr-challenge.git
%cd asr-challenge
!pip install -q huggingface_hub pyarrow soundfile librosa silero-vad scipy
!df -h /content | tail -1

## 1. 원본 음성 추출 (id 목록 재사용)

In [ ]:
!python data/scripts/select_subset.py --source zeroth
!python data/scripts/select_subset.py --source fleurs

## 2. 잡음 클립 (FSD50K, CC0·CC BY만)

In [ ]:
!python data/scripts/filter_noise.py
!tail -3 data/noise_list.csv

## 3. 합성

In [ ]:
!python data/scripts/mix_noise.py
!ls data/eval/audio && du -sh data/eval/audio
!python - <<'EOF'
import json, collections
rows=[json.loads(l) for l in open('data/eval/manifest.jsonl')]
print(len(rows), collections.Counter(r['condition'] for r in rows))
EOF

## 4. Hugging Face Datasets 업로드

`HF_TOKEN`(write 권한)을 Colab 보안 비밀에 넣어 두고 실행. 레포 id는 본인 계정으로 바꾼다.

In [ ]:
from google.colab import userdata
import os
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
!python data/scripts/upload_hf.py --repo ekgP525/campusvoice-rt-eval

## 5. manifest·잡음 목록을 레포로 되가져오기

아래 파일만 로컬 레포에 복사해 커밋하면 된다 (오디오는 HF에 있음).

In [ ]:
from google.colab import files
!zip -q -r eval_meta.zip data/eval/manifest.jsonl data/noise_list.csv data/LICENSES
files.download('eval_meta.zip')